# 예측 예제 및 실습

이 노트북은 **`협업 필터링`, `컨텐츠 기반 추천` 기초 예제 실습**을 수행하는 노트북입니다.

✅ 내용  
1. Suprise를 활용한 협업 필터링 예제
2. TF-IDF + 코사인 유사도를 활용한 컨텐츠 기반 추천 예제


In [ ]:
%conda env create -f recommendation.yaml -n recommendation

### Suprise를 활용한 협업 필터링 예제

MovieLens 100K 데이터셋을 활용하여 다양한 협업 필터링 알고리즘의 성능을 비교하고 개인화된 영화 추천을 생성합니다.

사용자-아이템 평점 데이터를 기반으로 KNN, SVD, SVD++ 모델을 학습하여 최적의 추천 알고리즘을 선정합니다.

* MovieLens 100K 데이터셋 자동 다운로드 및 영화 메타데이터(ID-제목) 매핑 처리
* KNNBasic(사용자 기반), SVD, SVD++ 세 가지 협업 필터링 알고리즘 동시 학습 및 비교
* RMSE 메트릭을 통한 모델별 예측 정확도 정량적 평가 및 성능 순위 산출
* 특정 사용자에 대한 개별 영화 예측 평점 계산 및 알고리즘별 예측값 비교 분석
* Top-N 추천 목록 생성으로 각 모델의 실제 추천 결과를 영화 제목과 함께 직관적 시각화


In [1]:
import os
import pandas as pd
from surprise import Dataset, Reader, KNNBasic, SVD, SVDpp, accuracy
from surprise.model_selection import train_test_split

# ----------------------------------------------------
# 0) 데이터 로드 (prompt=False로 묻지 않고 바로 다운로드)
# ----------------------------------------------------
data = Dataset.load_builtin('ml-100k', prompt=False)


# ----------------------------------------------------
# 1) 메타데이터 로드: MovieLens 100K u.item → ID→제목 매핑
# ----------------------------------------------------
base_path = os.path.expanduser('~/.surprise_data/ml-100k/ml-100k')
item_file = os.path.join(base_path, 'u.item')
items_df = pd.read_csv(
    item_file,
    sep='|',
    header=None,
    usecols=[0, 1],
    names=['item_id', 'title'],
    encoding='latin-1'
)

# ----------------------------------------------------
# 2) 훈련/테스트 분할
# ----------------------------------------------------
trainset, testset = train_test_split(data, test_size=0.25, random_state=42)

# ----------------------------------------------------
# 3) 모델 학습
# ----------------------------------------------------
algos = {
    'KNNBasic': KNNBasic(sim_options={'user_based': True}),
    'SVD':      SVD(),
    'SVD++':    SVDpp()
}

for name, algo in algos.items():
    algo.fit(trainset)

# ----------------------------------------------------
# 4) 모델별 RMSE 평가
# ----------------------------------------------------
print("=== RMSE 비교 ===")
for name, algo in algos.items():
    preds = algo.test(testset)
    rmse = accuracy.rmse(preds, verbose=False)
    print(f"{name:7s} : RMSE = {rmse:.4f}")
print()

# ----------------------------------------------------
# 5) 사용자별 예측 평점 직접 보기 (ID + 제목)
# ----------------------------------------------------
user_id = '196'
sample_items = ['302', '285', '500']  # 예시 아이템 몇 개
print(f"=== User {user_id}의 특정 영화 예측 평점 ===")
for name, algo in algos.items():
    print(f"\n[{name}]")
    for item_id in sample_items:
        pred = algo.predict(user_id, item_id)
        # 제목 매핑
        title = items_df.loc[items_df['item_id'] == int(item_id), 'title'].values[0]
        print(f"  Movie {item_id:3s} — {title:<40s} 예측평점: {pred.est:.2f}")
print()

# ----------------------------------------------------
# 6) Top-5 추천 목록 출력 (ID + 제목 + 예측평점)
# ----------------------------------------------------
def get_top_n_recs(algo, user_id, trainset, n=5):
    # 사용자가 평가하지 않은 모든 영화 목록
    anti_test = trainset.build_anti_testset()
    user_anti = [x for x in anti_test if x[0] == user_id]
    preds = algo.test(user_anti)
    # 예측 평점(est) 기준 내림차순 정렬 후 상위 n개
    top_n = sorted(preds, key=lambda x: x.est, reverse=True)[:n]
    return [(int(p.iid), p.est) for p in top_n]

print(f"=== User {user_id}에 대한 Top-5 추천 ===")
for name, algo in algos.items():
    recs = get_top_n_recs(algo, user_id, trainset, n=5)
    print(f"\n[{name}] 추천 영화:")
    for movie_id, est in recs:
        title = items_df.loc[items_df['item_id'] == movie_id, 'title'].values[0]
        print(f"  • {movie_id:4d} — {title:<40s} 예측평점: {est:.2f}")

Computing the msd similarity matrix...
Done computing similarity matrix.
=== RMSE 비교 ===
KNNBasic : RMSE = 0.9854
SVD     : RMSE = 0.9414
SVD++   : RMSE = 0.9248

=== User 196의 특정 영화 예측 평점 ===

[KNNBasic]
  Movie 302 — L.A. Confidential (1997)                 예측평점: 3.91
  Movie 285 — Secrets & Lies (1996)                    예측평점: 4.28
  Movie 500 — Fly Away Home (1996)                     예측평점: 4.03

[SVD]
  Movie 302 — L.A. Confidential (1997)                 예측평점: 4.10
  Movie 285 — Secrets & Lies (1996)                    예측평점: 4.05
  Movie 500 — Fly Away Home (1996)                     예측평점: 3.76

[SVD++]
  Movie 302 — L.A. Confidential (1997)                 예측평점: 4.16
  Movie 285 — Secrets & Lies (1996)                    예측평점: 4.11
  Movie 500 — Fly Away Home (1996)                     예측평점: 4.06

=== User 196에 대한 Top-5 추천 ===

[KNNBasic] 추천 영화:
  • 1189 — Prefontaine (1997)                       예측평점: 5.00
  • 1122 — They Made Me a Criminal (1939)           예측평점: 5.00
  • 1491 

### TF-IDF + 코사인 유사도를 활용한 컨텐츠 기반 추천 예제

Kaggle Amazon 상품 데이터셋을 활용하여 TF-IDF 벡터화와 코사인 유사도 기반의 내용 기반 필터링 추천 시스템을 구현합니다.

상품명, 카테고리, 설명 정보를 결합한 텍스트 특성으로부터 사용자 선호 상품과 유사한 아이템을 자동 발견하여 개인화된 쇼핑 추천을 제공합니다.

* Kaggle Hub를 통한 Amazon Sales Dataset 자동 다운로드 및 상품 메타데이터 전처리
* 상품명, 카테고리, 상품 설명을 결합한 통합 텍스트 특성 생성 및 결측치 처리
* TF-IDF 벡터화를 통한 텍스트 데이터의 수치적 표현 변환 및 고차원 특성 추출
* 모든 상품 쌍에 대한 코사인 유사도 행렬 사전 계산으로 실시간 추천 성능 최적화
* 다중 선호 상품 입력 지원 및 유사도 평균화를 통한 종합적 선호도 기반 Top-N 추천 생성

In [3]:
import kagglehub
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import numpy as np

# 1. 데이터셋 로드: 상품 데이터 CSV 파일 읽기
# Kaggle의 아마존 상품 데이터 'amazon.csv' 파일을 사용
# https://www.kaggle.com/datasets/karkavelrajaj/amazon-sales-dataset

path = kagglehub.dataset_download("karkavelrajaj/amazon-sales-dataset")
print("Path to dataset files:", path)  # 다운로드된 파일 경로 출력
path = path + "\\amazon.csv"  # Windows 경로 구분자 사용하여 CSV 파일 지정

df = pd.read_csv(path)  
# 데이터셋은 'name', 'category', 'description' 등의 열을 포함한다고 가정
# 결측치 처리 및 상품명/카테고리/설명 결합 (텍스트 기반 특성 생성)
df['product_name'] = df['product_name'].fillna('')
df['category'] = df['category'].fillna('')
df['about_product'] = df['about_product'].fillna('')
df['combined_features'] = df['product_name'] + ' ' + df['category'] + ' ' + df['about_product']

# 2. TF-IDF 벡터화: 텍스트 특성을 수치 벡터로 변환
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['combined_features'])
# tfidf_matrix는 각 상품의 TF-IDF 벡터 표현이며, 행 개수 = 상품 개수, 열 개수 = 단어 수

# 3. 코사인 유사도 계산: 모든 상품 쌍간 유사도 미리 계산
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
# cosine_sim는 (상품 수 x 상품 수) 크기의 배열로, cosine_sim[i][j]는 상품 i와 j 간의 코사인 유사도

# 4. 추천 함수 정의: 사용자가 좋아한 상품들을 입력받아 유사한 상품 추천
def recommend_similar(liked_indices, top_n=5):
    # 여러 상품을 좋아한 경우 유사도 벡터의 평균을 사용 (선호 아이템들의 종합적인 선호도)
    if len(liked_indices) == 0:
        return []
    if len(liked_indices) == 1:
        sim_scores = list(enumerate(cosine_sim[liked_indices[0]]))
    else:
        # liked_indices에 해당하는 여러 행의 유사도를 평균하여 단일 유사도 벡터 생성
        sim_scores = list(enumerate(np.mean(cosine_sim[liked_indices], axis=0)))
    # 자신이 선택한 상품들은 추천 목록에서 제외
    sim_scores = [score for score in sim_scores if score[0] not in liked_indices]
    # 유사도 점수를 기준으로 내림차순 정렬
    sim_scores.sort(key=lambda x: x[1], reverse=True)
    # 상위 top_n개의 상품 선택
    top_recs = sim_scores[:top_n]
    # 상품명과 유사도 점수로 결과 구성
    recommendations = [(df.loc[idx, 'product_name'], score) for idx, score in top_recs]
    return recommendations

# 5. 사용자 입력을 받아 추천 실행
keyword = 'SSD'
matched = df[df['product_name'].str.lower().str.contains(keyword.lower())]
liked_indices = [matched.index[0]] if not matched.empty else []

# 6. 추천 출력
if not liked_indices:
    print("선택한 상품이 없습니다. 조건을 확인해주세요.")
else:
    results = recommend_similar(liked_indices, top_n=5)
    print(f"\n[추천 결과] 선택한 상품과 유사한 Top 5:")
    for name, score in results:
        print(f"• {name:<40s}  (유사도: {score:.2f})")

Path to dataset files: C:\Users\USER\.cache\kagglehub\datasets\karkavelrajaj\amazon-sales-dataset\versions\1

[추천 결과] 선택한 상품과 유사한 Top 5:
• Crucial P3 500GB PCIe 3.0 3D NAND NVMe M.2 SSD, up to 3500MB/s - CT500P3SSD8  (유사도: 0.29)
• Western Digital WD Green SATA 240GB Internal SSD Solid State Drive - SATA 6Gb/s 2.5 inches - WDS240G3G0A  (유사도: 0.22)
• Lapster USB 3.0 sata Cable for 2.5 inch SSD and HDD , USB 3.0 to SATA III Hard Driver Adapter , sata to USB Cable-(Blue)  (유사도: 0.22)
• CARECASE® Optical Bay 2nd Hard Drive Caddy, 9.5 mm CD/DVD Drive Slot for SSD and HDD  (유사도: 0.20)
• Cablet 2.5 Inch SATA USB 3.0 HDD/SSD Portable External Enclosure for 7mm and 9.5mm, Tool-Free Design, Supports UASP Max 6TB  (유사도: 0.19)
